##**Personalized TED Talks Recommendation System using Machine Learning**

---



*1. Dataset Import :*

In [7]:
import pandas as pd
import numpy as np

file_path = "/content/TED_Talks_Dataset.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully!")
print("Original Dataset Shape:", df.shape)

display(df.head())



Dataset loaded successfully!
Original Dataset Shape: (4082, 406)


,Talk_Id,Title,Speaker_1,All_Speakers,Occupations,About_speakers,Views,Recorded_date,Published_date,Event,...,Unnamed: 396,Unnamed: 397,Unnamed: 398,Unnamed: 399,Unnamed: 400,Unnamed: 401,Unnamed: 402,Unnamed: 403,Unnamed: 404,Unnamed: 405
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,25-02-2006,27-06-2006,TED2006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,22-02-2006,27-06-2006,TED2006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,24-02-2006,27-06-2006,TED2006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,26-02-2006,27-06-2006,TED2006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,25-02-2006,27-06-2006,TED2006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


*2. Data Quality Report Before Cleaning:*

In [ ]:

print("DATA QUALITY REPORT - BEFORE CLEANING")
print("=" * 60)

print("\n1. Dataset Information :")
print("-" * 40)

print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

missing_values = df.isna().sum().sum()

print("\n2. Missing Values :")
print("-" * 40)

print("Total Missing Values:", missing_values)

print("\nMissing Values by Column:")
display(df.isna().sum().to_frame("Missing Count"))

duplicate_count = df.duplicated().sum()

print("\n3. Duplicate Records :")
print("-" * 40)

print("Total Duplicate Rows:", duplicate_count)

whitespace_count = df.apply(
    lambda col: col.map(
        lambda value: isinstance(value, str)
        and value != ""
        and value.isspace()
    ).sum()
).sum()

print("\n4. Empty Spaces :")
print("-" * 40)

print("Whitespace-Only Cells:", whitespace_count)

numeric_columns = [
    col for col in ["talk_id", "views", "comments", "duration"]
    if col in df.columns
]

numeric_error_count = 0

for col in numeric_columns:
    values = df[col].astype("string").str.strip()

    # Ignore missing and whitespace-only values.
    valid_entries = values.notna() & values.ne("")

    converted = pd.to_numeric(values, errors="coerce")

    # Count non-empty values that cannot be converted to numbers.
    errors = valid_entries & converted.isna()

    numeric_error_count += errors.sum()

print("\n5. Invalid Numeric Values :")
print("-" * 40)

print("Total Invalid Numeric Entries:", numeric_error_count)

# Invalid date values
date_columns = [
    col for col in ["recorded_date", "published_date"]
    if col in df.columns
]

date_error_count = 0

for col in date_columns:
    values = df[col].astype("string").str.strip()

    valid_entries = values.notna() & values.ne("")

    converted = pd.to_datetime(
        values,
        format="%d-%m-%Y",
        errors="coerce"
    )

    errors = valid_entries & converted.isna()

    date_error_count += errors.sum()

print("\n6. Invalid Date Values :")
print("-" * 40)

print("Total Invalid Date Entries:", date_error_count)

unnamed_columns = [
    col for col in df.columns
    if str(col).startswith("Unnamed:")
]

print("\n7. Unnecessary Columns :")
print("-" * 40)

print("Number of Unnamed Columns:", len(unnamed_columns))


print("BEFORE CLEANING SUMMARY")
print("=" * 60)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing Values:", missing_values)
print("Duplicate Rows:", duplicate_count)
print("Whitespace-Only Cells:", whitespace_count)
print("Invalid Numeric Entries:", numeric_error_count)
print("Invalid Date Entries:", date_error_count)
print("Unnecessary Columns:", len(unnamed_columns))

DATA QUALITY REPORT - BEFORE CLEANING

1. Dataset Information :
----------------------------------------
Number of Rows: 4049
Number of Columns: 406

2. Missing Values :
----------------------------------------
Total Missing Values: 1567295

Missing Values by Column:


,Missing Count
Talk_Id,33
Title,34
Speaker_1,34
All_Speakers,38
Occupations,557
...,...
Unnamed: 401,4048
Unnamed: 402,4048
Unnamed: 403,4048
Unnamed: 404,4048



3. Duplicate Records :
----------------------------------------
Total Duplicate Rows: 32

4. Empty Spaces :
----------------------------------------
Whitespace-Only Cells: 0

5. Invalid Numeric Values :
----------------------------------------
Total Invalid Numeric Entries: 0

6. Invalid Date Values :
----------------------------------------
Total Invalid Date Entries: 0

7. Unnecessary Columns :
----------------------------------------
Number of Unnamed Columns: 387
BEFORE CLEANING SUMMARY
Rows: 4049
Columns: 406
Missing Values: 1567295
Duplicate Rows: 32
Whitespace-Only Cells: 0
Invalid Numeric Entries: 0
Invalid Date Entries: 0
Unnecessary Columns: 387


*3. Data Cleaning :*

In [ ]:
df_clean = df.copy()

print("STARTING DATA CLEANING..........")
unnamed_columns = [
    col for col in df_clean.columns
    if str(col).startswith("Unnamed:")
]

df_clean.drop(columns=unnamed_columns, inplace=True)

print("\n1. Unnecessary columns removed :", len(unnamed_columns))

df_clean.columns = df_clean.columns.str.strip()

print("2. Column names cleaned :")

for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[col] = df_clean[col].map(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

print("3. Leading and trailing spaces removed :")

df_clean.replace(r"^\s*$", np.nan, regex=True, inplace=True)

print("4. Empty strings replaced with NaN :")

numeric_columns = [
    col for col in ["talk_id", "views", "comments", "duration"]
    if col in df_clean.columns
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

print("5. Numeric columns converted :")

date_columns = [
    col for col in ["recorded_date", "published_date"]
    if col in df_clean.columns
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        format="%d-%m-%Y",
        errors="coerce"
    )

print("6. Date columns converted :")

duplicates_before = df_clean.duplicated().sum()

df_clean.drop_duplicates(inplace=True)

print("7. Duplicate rows removed :", duplicates_before)

empty_rows_before = df_clean.isna().all(axis=1).sum()

df_clean.dropna(how="all", inplace=True)

print("8. Completely empty rows removed :", empty_rows_before)

df_clean.reset_index(drop=True, inplace=True)

print("9. Dataset index reset :")


print("\n")
print("DATA CLEANING COMPLETED...!")

print("\nOriginal Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)

display(df_clean.head())

STARTING DATA CLEANING..........

1. Unnecessary columns removed : 387
2. Column names cleaned :
3. Leading and trailing spaces removed :
4. Empty strings replaced with NaN :
5. Numeric columns converted :
6. Date columns converted :
7. Duplicate rows removed : 32
8. Completely empty rows removed : 1
9. Dataset index reset :


DATA CLEANING COMPLETED...!

Original Shape: (4049, 406)
Cleaned Shape: (4016, 19)


,Talk_Id,Title,Speaker_1,All_Speakers,Occupations,About_speakers,Views,Recorded_date,Published_date,Event,Native_lang,Available_lang,Comments,Duration,Topics,Related_talks,URL,Description,Transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,25-02-2006,27-06-2006,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,22-02-2006,27-06-2006,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,24-02-2006,27-06-2006,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,26-02-2006,27-06-2006,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,25-02-2006,27-06-2006,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


*4. Handling Missing Values :*

In [ ]:
print("=" * 60)
print("MISSING VALUE HANDLING")
print("=" * 60)

print("\nMissing Values Before Handling:")

display(
    df_clean.isna()
    .sum()
    .to_frame("Missing Count")
    .query("`Missing Count` > 0")
)

# Fill missing text values with a placeholder
text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df_clean[col] = df_clean[col].fillna("Not Available")

# Fill missing numeric values with the median
numeric_columns = [
    col for col in df_clean.select_dtypes(include=["number"]).columns
    if col != "talk_id"
]

for col in numeric_columns:
    if df_clean[col].isna().any():
        median_value = df_clean[col].median()

        if pd.notna(median_value):
            df_clean[col] = df_clean[col].fillna(median_value)

# Leave missing dates as NaT rather than inventing dates.

print("\nMissing Values After Handling:")

display(
    df_clean.isna()
    .sum()
    .to_frame("Missing Count")
    .query("`Missing Count` > 0")
)

print("\nMissing value handling completed!")

MISSING VALUE HANDLING

Missing Values Before Handling:


,Missing Count
Title,1
Speaker_1,1
All_Speakers,5
Occupations,524
About_speakers,505
Views,2
Recorded_date,3
Published_date,2
Event,2
Native_lang,2



Missing Values After Handling:


,Missing Count



Missing value handling completed!


*5. Data Quality Report after Cleaning :*

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT - AFTER CLEANING")
print("=" * 60)

print("\n1. Dataset Information")
print("-" * 40)

print("Number of Rows:", df_clean.shape[0])
print("Number of Columns:", df_clean.shape[1])

print("\n2. Missing Values")
print("-" * 40)

print("Total Missing Values:", df_clean.isna().sum().sum())

print("\nMissing Values by Column:")

display(
    df_clean.isna()
    .sum()
    .to_frame("Missing Count")
    .query("`Missing Count` > 0")
)

print("\n3. Duplicate Records")
print("-" * 40)

print("Total Duplicate Rows:", df_clean.duplicated().sum())

print("\n4. Empty Spaces")
print("-" * 40)

whitespace_after = df_clean.apply(
    lambda col: col.map(
        lambda value: isinstance(value, str)
        and value != ""
        and value.isspace()
    ).sum()
).sum()

print("Whitespace-Only Cells:", whitespace_after)

print("\n5. Data Types")
print("-" * 40)

display(df_clean.dtypes.to_frame("Data Type"))

print("\n" + "=" * 60)
print("AFTER CLEANING SUMMARY")
print("=" * 60)

print("Original Rows:", df.shape[0])
print("Cleaned Rows:", df_clean.shape[0])

print("Original Columns:", df.shape[1])
print("Cleaned Columns:", df_clean.shape[1])

print("Duplicate Rows Remaining:", df_clean.duplicated().sum())

print("Missing Values Remaining:", df_clean.isna().sum().sum())

print("Whitespace-Only Cells Remaining:", whitespace_after)

DATA QUALITY REPORT - AFTER CLEANING

1. Dataset Information
----------------------------------------
Number of Rows: 4016
Number of Columns: 19

2. Missing Values
----------------------------------------
Total Missing Values: 0

Missing Values by Column:


,Missing Count



3. Duplicate Records
----------------------------------------
Total Duplicate Rows: 0

4. Empty Spaces
----------------------------------------
Whitespace-Only Cells: 0

5. Data Types
----------------------------------------


,Data Type
Talk_Id,object
Title,object
Speaker_1,object
All_Speakers,object
Occupations,object
About_speakers,object
Views,object
Recorded_date,object
Published_date,object
Event,object



AFTER CLEANING SUMMARY
Original Rows: 4049
Cleaned Rows: 4016
Original Columns: 406
Cleaned Columns: 19
Duplicate Rows Remaining: 0
Missing Values Remaining: 0
Whitespace-Only Cells Remaining: 0


*7. Before and After Comparison :*

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Number of Rows",
        "Number of Columns",
        "Missing Values",
        "Duplicate Rows",
        "Whitespace-Only Cells",
        "Unnecessary Columns"
    ],

    "Before Cleaning": [
        df.shape[0],
        df.shape[1],
        df.isna().sum().sum(),
        df.duplicated().sum(),

        df.apply(
            lambda col: col.map(
                lambda value: isinstance(value, str)
                and value != ""
                and value.isspace()
            ).sum()
        ).sum(),

        sum(
            str(col).startswith("Unnamed:")
            for col in df.columns
        )
    ],

    "After Cleaning": [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isna().sum().sum(),
        df_clean.duplicated().sum(),

        df_clean.apply(
            lambda col: col.map(
                lambda value: isinstance(value, str)
                and value != ""
                and value.isspace()
            ).sum()
        ).sum(),

        sum(
            str(col).startswith("Unnamed:")
            for col in df_clean.columns
        )
    ]
})

display(comparison)

,Metric,Before Cleaning,After Cleaning
0,Number of Rows,4049,4016
1,Number of Columns,406,19
2,Missing Values,1567295,0
3,Duplicate Rows,32,0
4,Whitespace-Only Cells,0,0
5,Unnecessary Columns,387,0


*8. Export Cleaned Dataset :*

In [ ]:
output_file = "TED_Talks_Cleaned.csv"

df_clean.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully!")
print("File Name:", output_file)

# Download the cleaned CSV
files.download(output_file)

Cleaned dataset saved successfully!
File Name: TED_Talks_Cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>